# Verify rung 0 yourself

This notebook re-derives the rung-0 result — the reproducibility ceiling of drug-response
profiles in the Tahoe-100M screen — from the files committed in this repository. It
starts from the result itself: every section first **shows** the thing it is about to
check (the table, the data, the distribution), then checks it in plain sight with
standard-library hashing, direct `pandas` reads, and explicit arithmetic. Nothing is
imported from this repository's own code, no stored image is displayed, and every figure
is drawn live from the same data frames the checks just used. Run all cells (Run → Run
All Cells; about a minute on a laptop).

To run it: from the repository root, `uv sync --extra dev`, then
`uv run jupyter lab docs/tasks/rung0-replicate-ceiling/verify.ipynb`.

**What cannot be checked on a laptop, stated rather than hidden:** the gene and drug
panel files live on the Alpine cluster, pinned by checksum in the run record, so the
declared panel size (14,121 genes) is a recorded input property here, not a recomputable
one; the 1,026 data files sit on cluster scratch, so their integrity reduces locally to
the committed inventory hashing to the data version the run record pinned (section 2);
and the 1,600 per-condition correlation values live only in their summary row — their
full histogram is produced by the cluster run — so the distributions drawn below come
from the committed per-gene, per-shuffle, and screen-composition tables.

In [ ]:
import hashlib
import json
import re
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert repo is not None, 'run this notebook from inside the repository (its own folder works)'
task = repo / 'docs' / 'tasks' / 'rung0-replicate-ceiling'
results = repo / 'results' / 'rung0-replicate-ceiling'
print('repository:', repo)

## 1. The result, and that it is unchanged since it was computed

The whole analysis produces one row of numbers: for 1,600 (cell line, drug) conditions,
how well two independent halves of the replicate plates agree with each other — the
reproducibility of the drug response itself, which caps what any prediction of it could
score. Read that row, then the figure places every reported number on one axis: the
middle half of conditions, the median and mean, the two chance floors (how well
*mismatched* conditions agree), the two detection thresholds (the smallest effect the
analysis had 80% power to find), and the Spearman-Brown estimate of full-data
reliability — the ceiling every later model score is read against.

In [ ]:
headline = pd.read_csv(results / 'rung0_delta_reproducibility.csv').iloc[0]
print(headline.to_string())

m = headline['splithalf_mean_r']
# the orderings the figure should show, checked before drawing it
assert headline['splithalf_q1_r'] <= headline['splithalf_median_r'] <= headline['splithalf_q3_r']
assert headline['null_diff_drug_mean_r'] < headline['null_same_drug_mean_r'] < m
assert headline['mde_80_vs_diff_drug'] < m and headline['mde_80_vs_same_drug'] < m

fig, ax = plt.subplots(figsize=(9.5, 3))
ax.axvspan(headline['splithalf_q1_r'], headline['splithalf_q3_r'], color='0.87',
           label=f"middle half of conditions ({headline['splithalf_q1_r']}-{headline['splithalf_q3_r']})")
ax.axvline(headline['splithalf_median_r'], color='0.35', lw=1.5,
           label=f"median condition {headline['splithalf_median_r']}")
ax.axvline(m, color='crimson', lw=2.2, label=f'mean over conditions {m} (the headline)')
ax.axvline(headline['spearman_brown_full'], color='purple', lw=2.2,
           label=f"Spearman-Brown full-data ceiling {headline['spearman_brown_full']}")
ax.axvline(headline['null_same_drug_mean_r'], ls='--', color='darkorange',
           label=f"same-drug chance floor {headline['null_same_drug_mean_r']}")
ax.axvline(headline['null_diff_drug_mean_r'], ls='--', color='steelblue',
           label=f"mismatched chance floor {headline['null_diff_drug_mean_r']}")
ax.axvline(headline['mde_80_vs_same_drug'], ls=':', color='darkorange',
           label=f"detection threshold vs same-drug {headline['mde_80_vs_same_drug']}")
ax.axvline(headline['mde_80_vs_diff_drug'], ls=':', color='steelblue',
           label=f"detection threshold vs mismatched {headline['mde_80_vs_diff_drug']}")
ax.set_yticks([])
ax.set_xlim(0, 0.27)
ax.set_xlabel('agreement between independent half-measurements (Pearson r)')
ax.set_title('The rung-0 result on one axis')
ax.legend(frameon=False, fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
plt.show()

That table is the file this repository cites everywhere else. When the analysis ran, a
record was written beside it (`rung0_delta_reproducibility.provenance.json`) carrying the
file's SHA-256 fingerprint, the producing commit, and the job that made it. Recompute the
fingerprint from the file you just read: if it matches, the numbers above cannot have
been edited since the analysis wrote them. Same for the cluster job's log, and the
working copy in this folder must be byte-identical to the cited copy.

In [ ]:
record = json.loads((results / 'rung0_delta_reproducibility.provenance.json').read_text())

result_sha = hashlib.sha256((repo / record['result']).read_bytes()).hexdigest()
print('fingerprint recorded at run time :', record['result_sha256'])
print('fingerprint of the file above    :', result_sha)
assert result_sha == record['result_sha256']

log_sha = hashlib.sha256((repo / record['log']).read_bytes()).hexdigest()
print('job-log fingerprint, recorded    :', record['log_sha256'])
print('job-log fingerprint, recomputed  :', log_sha)
assert log_sha == record['log_sha256']

assert (task / 'rung0_delta_reproducibility.csv').read_bytes() == (repo / record['result']).read_bytes()
print('the working copy in this folder is byte-identical to the cited copy')

## 2. The data, and that the screen contains what the analysis needs

The input is the Tahoe-100M screen's differential-expression table: for each cancer cell
line and drug, the per-gene expression change (log2 fold change) of treated cells against
plate-matched solvent controls, measured on replicate plates. It was downloaded
2026-07-24 as 1,026 files and restricted to the 32 drugs shared with the GDSC2 drug-
sensitivity screen (Genomics of Drug Sensitivity in Cancer, release 2) across 50 cell
lines. For the analysis to be meaningful, the download and processing had to deliver:
**every (line, drug) condition of that panel present, with replicate plates to split and
a uniform dose design.** The run measured what it actually consumed into a composition
table — verify the goal was met from that table, and draw the delivered screen: the full
drug-by-line grid, colored by replicate depth. A complete grid with no holes is the
download goal, visibly achieved; the one single-plate row (Ribociclib) is the known
exception the analysis must — and does, section 4 — handle.

In [ ]:
# keep_default_na: one line key is literally the string 'NA' (a missing DepMap id
# carried through from the source) and must stay text, not become missing data
pool = pd.read_csv(task / 'rung0_pool_description.csv', keep_default_na=False)

print(len(pool), 'conditions:', pool['patient'].nunique(), 'cell lines x',
      pool['drug'].nunique(), 'drug names (32 declared drugs + one solvate name variant)')
print('drugs delivered:', ', '.join(sorted(pool['drug'].unique())))
assert len(pool) == 1650 and pool['patient'].nunique() == 50 and pool['drug'].nunique() == 33
assert (pool['n_dose_levels'] == 3).all(), 'dose design must be uniform: 3 levels everywhere'
print('every condition carries exactly 3 dose levels')

plates = pool.pivot(index='drug', columns='patient', values='n_plates')
assert plates.notna().all().all(), 'the drug x line grid must be complete -- no missing conditions'
assert plates.shape == (33, 50)

fig, ax = plt.subplots(figsize=(11, 6.8))
im = ax.pcolormesh(plates.to_numpy(), cmap=plt.get_cmap('viridis', 7), vmin=0.5, vmax=7.5)
ax.set_yticks(np.arange(len(plates.index)) + 0.5)
ax.set_yticklabels(plates.index, fontsize=7)
ax.set_xticks([])
ax.set_xlabel(f'{plates.shape[1]} cell lines')
ax.set_title('The delivered screen: complete 33 x 50 grid, colored by replicate plates')
cbar = fig.colorbar(im, ticks=range(1, 8))
cbar.set_label('replicate plates')
fig.tight_layout()
plt.show()

And the data cannot have changed since: the repository commits an inventory of all 1,026
downloaded files (path, size, SHA-256 checksum — one line each). The checksum of that
inventory is the data version the run record pinned, so the bytes the analysis read are
fixed by files you can hash yourself.

In [ ]:
manifest = repo / 'data' / 'tranches' / 'tahoe100m-pseudobulk-de.v1.manifest.txt'
registration = json.loads((repo / 'data' / 'tranches' / 'tahoe100m-pseudobulk-de.v1.json').read_text())

manifest_sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
print('files in the inventory              :', len(manifest.read_text().splitlines()))
print('checksum of the inventory           :', manifest_sha)
print('data version at registration        :', registration['content_hash'])
print('data version the run record pinned  :', record['environment']['data_commit'])
assert len(manifest.read_text().splitlines()) == 1026
assert manifest_sha == registration['content_hash'] == record['environment']['data_commit']

## 3. The two headline claims, each drawn

**Claim: the full-data ceiling is 0.238.** The analysis splits the plates in half, so the
measured agreement (0.135) understates what the complete data supports; the Spearman-
Brown formula 2r/(1+r) converts half-data agreement into the full-data estimate. Draw the
whole conversion curve and place the measurement on it — the reported ceiling must be the
curve's value at the measured point, and you can read from the curve what any other
measurement would have implied.

In [ ]:
print('measured half-data agreement :', m)
print('2r/(1+r)                     =', round(2 * m / (1 + m), 3))
print('reported full-data ceiling   :', headline['spearman_brown_full'])
assert round(2 * m / (1 + m), 3) == headline['spearman_brown_full']

r = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(r, 2 * r / (1 + r), color='0.3', label='Spearman-Brown: full = 2r/(1+r)')
ax.plot([0, 1], [0, 1], ls=':', color='0.7', label='no correction')
ax.plot(m, headline['spearman_brown_full'], 'o', color='crimson', ms=8, zorder=5,
        label=f"this measurement ({m} -> {headline['spearman_brown_full']})")
ax.plot([m, m, 0], [0, headline['spearman_brown_full'], headline['spearman_brown_full']],
        color='crimson', lw=0.8, ls='--')
ax.set_xlabel('split-half agreement (measured)')
ax.set_ylabel('full-data reliability (estimated)')
ax.set_title('The ceiling is the curve at the measured point')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

**Claim: reliability rises with effect size** — the built-in positive control. Conditions
were split into thirds by how large their expression response is; an assay that could not
find more reproducibility where there is more signal would be broken. The bars must rise,
and every tercile must clear both chance floors.

In [ ]:
terciles = [headline[f'splithalf_mean_r_tercile{t}'] for t in (1, 2, 3)]
print('tercile reliabilities, weakest to strongest responses:', terciles)
assert terciles[0] < terciles[1] < terciles[2]
assert terciles[0] > headline['null_same_drug_mean_r'] > headline['null_diff_drug_mean_r']

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(['weakest third\n(by effect size)', 'middle third', 'strongest third'], terciles,
       color='0.7', edgecolor='0.3')
ax.axhline(m, color='crimson', lw=1.8, label=f'overall mean {m}')
ax.axhline(headline['null_same_drug_mean_r'], ls='--', color='darkorange',
           label=f"same-drug chance floor {headline['null_same_drug_mean_r']}")
ax.axhline(headline['null_diff_drug_mean_r'], ls='--', color='steelblue',
           label=f"mismatched chance floor {headline['null_diff_drug_mean_r']}")
ax.set_ylabel('split-half reliability (mean Pearson r)')
ax.set_title('More signal, more reproducibility -- and every tercile clears both floors')
ax.legend(frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
fig.tight_layout()
plt.show()

## 4. The 1,600 scored conditions are exactly the splittable ones

Section 2's grid showed one single-plate row. A single plate cannot be split in half, so
those conditions cannot be scored — and they must be exactly the 50 Ribociclib ones, with
1,650 − 50 equal to the result's condition count. The figure shows every plate-split
configuration; the red bar is the excluded 50.

In [ ]:
unsplittable = pool[(pool['n_plates_half0'] == 0) | (pool['n_plates_half1'] == 0)]
print(len(unsplittable), 'unsplittable conditions, drug(s):', sorted(unsplittable['drug'].unique()))
print('scored =', len(pool) - len(unsplittable), '  reported n_pairs =', int(headline['n_pairs']))
assert len(unsplittable) == 50 and set(unsplittable['drug']) == {'Ribociclib'}
assert len(pool) - len(unsplittable) == int(headline['n_pairs']) == 1600

config = pool.groupby(['n_plates_half0', 'n_plates_half1']).size().sort_index()
labels = [f'{a} + {b}' for a, b in config.index]
colors = ['crimson' if (a == 0 or b == 0) else '0.7' for a, b in config.index]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(labels, config.to_numpy(), color=colors, edgecolor='0.3')
for x, v in zip(labels, config.to_numpy()):
    ax.text(x, v, str(v), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('replicate plates in half 0 + half 1')
ax.set_ylabel('(line, drug) conditions')
ax.set_title('Plate-split configurations; red = unsplittable (all 50 are Ribociclib)')
fig.tight_layout()
plt.show()

## 5. The significance survives the dependence check

The reported p-values assume the mismatched-pair null draws behave like an exchangeable
pool although they reuse half-profiles. The task measured that assumption with shuffle
(derangement) checks — 500 permutations for the pooled comparison and for each of the
two comparison types the analysis reports, every per-permutation mean committed.
Recompute each null's mean and spread and the exact p = (1 + #{shuffles ≥ observed}) /
(1 + 500), and confirm the true pairing beats all 500 shuffles in every stratum. The
figure is the whole argument at a glance: the gray histogram is what mean agreement
looks like when the pairing is destroyed, and the red line is the true pairing.

In [ ]:
der = pd.read_csv(task / 'rung0_derangement_summary.csv').iloc[0]
strata = [
    ('any-pair shuffle', 'rung0_derangement_perm_means.csv', 'observed_mean', ''),
    ('same-drug shuffle', 'rung0_derangement_perm_means_same_drug.csv', 'observed_mean_same_drug_rows', '_same_drug'),
    ('diff-drug shuffle', 'rung0_derangement_perm_means_diff_drug.csv', 'observed_mean_diff_drug_rows', '_diff_drug'),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True)
for ax, (name, filename, observed_col, suffix) in zip(axes, strata):
    perms = pd.read_csv(task / filename)['perm_mean']
    observed = der[observed_col]
    p = (1 + int((perms >= observed).sum())) / (1 + len(perms))
    print(f'{name:18s}: {len(perms)} shuffles  mean {perms.mean():.4f}  sd {perms.std():.4f}  '
          f'max {perms.max():.4f} < observed {observed}  exact p {p:.3f}')
    assert len(perms) == 500 and observed > perms.max()
    assert round(perms.mean(), 4) == der['perm_mean_mean' + suffix]
    assert round(perms.std(), 4) == der['perm_mean_sd' + suffix]
    assert round(p, 3) == der['p_exact' + suffix]
    ax.hist(perms, bins=30, color='0.7', edgecolor='0.5')
    ax.axvline(observed, color='crimson', lw=2)
    ax.set_title(f'{name}\n500 shuffled nulls vs observed (p = {p:.3f})', fontsize=9)
    ax.set_xlabel('mean mismatched correlation')
axes[0].set_ylabel('shuffles')
fig.tight_layout()
plt.show()

The *design effect* is the shuffle-measured variance of the null mean divided by the
variance the exchangeable-pool shortcut assumed. Below one means the shortcut was
cautious, not generous. Recompute the pooled one from the committed draws (the
per-stratum ones need each stratum's pooled standard error, which only the cluster run
holds — they are transcribed in the summary and cross-checked in continuous
integration). Then confirm two entirely different sampling mechanisms — these
derangements and the headline's bootstrapped pools — land on the same chance floors.

In [ ]:
any_perms = pd.read_csv(task / 'rung0_derangement_perm_means.csv')['perm_mean']
design_effect = any_perms.var() / der['se_iid_pool'] ** 2
print('design effect =', round(design_effect, 3), '  reported:', der['design_effect'])
assert abs(design_effect - der['design_effect']) < 0.02 and design_effect < 1

print('same-drug floor: derangement', der['perm_mean_mean_same_drug'],
      ' bootstrap', headline['null_same_drug_mean_r'])
print('diff-drug floor: derangement', der['perm_mean_mean_diff_drug'],
      ' bootstrap', headline['null_diff_drug_mean_r'])
assert abs(der['perm_mean_mean_same_drug'] - headline['null_same_drug_mean_r']) < 0.0015
assert abs(der['perm_mean_mean_diff_drug'] - headline['null_diff_drug_mean_r']) < 0.0015

## 6. Reliability is broadly distributed, led by stress-response genes

The per-gene diagnostic (reported but not part of the headline): 13,886 panel genes, each
correlated across conditions between the two plate halves. Recompute the write-up's
numbers — 97.0% positive, median 0.146, quartiles 0.089–0.230 — and draw the
distribution: nearly the whole panel sits right of zero, and the extreme right tail is
heat-shock and immediate-early stress-response transcripts.

In [ ]:
pg = pd.read_csv(task / 'rung0_per_gene_reliability.csv')
finite = pg[np.isfinite(pg['r'])]
top5 = pg.nlargest(5, 'r')

print(len(pg), 'genes,', len(finite), 'with a finite value,', len(pg) - len(finite), 'without')
print(f"positive fraction {100 * (finite['r'] > 0).mean():.1f}%  "
      f"median {finite['r'].median():.3f}  "
      f"quartiles {finite['r'].quantile(0.25):.3f}-{finite['r'].quantile(0.75):.3f}")
print(top5.to_string(index=False))
assert len(pg) == 13886 and len(finite) == 13759
assert abs((finite['r'] > 0).mean() - 0.970) <= 0.0005
assert abs(finite['r'].median() - 0.146) <= 0.0005
assert abs(finite['r'].quantile(0.25) - 0.089) <= 0.0005
assert abs(finite['r'].quantile(0.75) - 0.230) <= 0.0005
assert list(top5['gene']) == ['HSP90AA1', 'EGR1', 'HSPA1B', 'HSPH1', 'PLEC']

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.hist(finite['r'], bins=90, color='0.7', edgecolor='0.6')
ax.axvline(0, color='k', lw=1)
ax.axvline(finite['r'].median(), color='crimson', lw=1.8,
           label=f"median {finite['r'].median():.3f}")
for (_, g), y in zip(top5.iterrows(), (900, 740, 580, 420, 260)):
    ax.plot([g['r'], g['r']], [0, y - 40], color='0.45', lw=0.6)
    ax.annotate(g['gene'], (g['r'], y), fontsize=8, ha='center')
ax.set_xlabel('per-gene split-half reliability r (across 1,600 conditions)')
ax.set_ylabel('genes')
ax.set_title('13,759 panel genes: 97.0% right of zero; stress-response genes lead')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 7. The summary says what the artifacts say

Parse the evidence table out of `summary.md` and compare each number to the artifact it
came from — a transcribed number that drifts from its artifact fails right here. (The
prose paragraphs' numbers — terciles, power ratios, per-stratum design effects — get
the same treatment in the continuous-integration battery; section 8 runs it.)

In [ ]:
text = (task / 'summary.md').read_text()

def numbers(label):
    line = next(l for l in text.splitlines() if l.startswith('|') and label in l)
    return [float(x.replace(',', '')) for x in re.findall(r'\d[\d,]*\.?\d*', line.rsplit('|', 2)[-2])]

assert numbers('Conditions scored') == [headline['n_pairs']]
assert numbers('Panel genes present')[0] == headline['n_genes']  # 14,121 is hash-pinned, see intro
assert numbers('Split-half reliability') == [headline['splithalf_mean_r'], headline['splithalf_median_r'],
                                             headline['splithalf_q1_r'], headline['splithalf_q3_r']]
assert numbers('Spearman-Brown') == [headline['spearman_brown_full']]
assert numbers('positive reliability') == [round(100 * headline['frac_pos'], 1)]
assert numbers('Mismatched-condition floor')[-1] == headline['null_diff_drug_mean_r']
assert numbers('Same-drug floor')[-1] == headline['null_same_drug_mean_r']
assert numbers('Significance')[0] == headline['p_vs_null'] == headline['p_vs_same_drug']
assert numbers('Smallest detectable')[-2:] == [round(headline['mde_80_vs_diff_drug'], 3),
                                              round(headline['mde_80_vs_same_drug'], 3)]
print('every evidence-table number matches its artifact')

## 8. The instruments themselves are validated

Everything above checks the numbers; these two cells check the code that produced them.
First the known-answer suite: synthetic data with a planted reliability of 0.8 must
come out 0.8 through the real measurement, a signal-free pool must come out null, the
power calculation must match the closed-form normal-theory answer, and the one
statistical defect this project has actually shipped (comparing an aggregate against
single draws) is pinned by a test demonstrating the wrong form failing. Then the full
continuous-integration battery (`scripts/verify_rung0.py` — the scripted form of the
cells above plus the prose-paragraph checks), which must agree with everything you just
watched.

In [ ]:
result = subprocess.run(['uv', 'run', 'pytest', '-m', 'known_answer', '-q'],
                        cwd=repo, capture_output=True, text=True)
print(result.stdout[-2500:] + result.stderr[-500:])
assert result.returncode == 0, 'known-answer controls failed'

In [ ]:
battery = subprocess.run(['uv', 'run', 'python', 'scripts/verify_rung0.py'],
                         cwd=repo, capture_output=True, text=True)
print(battery.stdout[-600:])
assert battery.returncode == 0, 'the scripted battery disagrees with the cells above'
print('You have re-derived rung 0, not read about it.')